# cancer-recon-apoptosis — RUNG 10 / arm (a): the GPU 3-input SURFACE AND-NOT sweep

The confirmatory run RUNG-6 promised. RUNG-5 found 0/1000 single+2-input surface gates worst-donor-safe.
**Does any 3-input SURFACE AND-NOT gate** (`posA AND posB AND-NOT negC`) close the gap — or does a negative
finally prove the NOT-slot **must** be a genetic-loss signal (HLA-LOH), not a surface marker?

**⚠️ This rung needs a GPU.** Unlike the fetch-bound rungs, scoring ~10⁵–10⁶ gates × 1.26M cells is
compute-bound — the T4 genuinely earns its place (CuPy boolean AND + bincount). **Runtime → Change runtime
type → T4 GPU.**

It **reuses your RUNG-5 atlas panel** (the `.r5.normal/.r5.tumour` caches on Drive) — if you ran RUNG-5 on
this account it loads instantly; if not, it re-fetches (resumable, same as RUNG-5). Resumable per-batch
checkpoints + heartbeat as always. Scorer = the **same audited `score_gates_vec`** RUNG-5 used (apples-to-apples).

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recon-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — mount Drive, reuse the RUNG-5 panel cache, enable GPU, start the run log
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    cache_dir = '/content/drive/MyDrive/cancer-recon'
    os.makedirs(cache_dir, exist_ok=True)
    # SAME path RUNG-5 used -> reuses .r5.normal/.r5.tumour panel caches (instant if present on THIS account)
    os.environ['LOGICGATE_CACHE'] = f'{cache_dir}/rung5_cache.npz'
    print('[CELL 2] Drive mounted. LOGICGATE_CACHE =', os.environ['LOGICGATE_CACHE'])
    import os.path as _p
    has = _p.exists(f'{cache_dir}/rung5_cache.r5.normal.npz') and _p.exists(f'{cache_dir}/rung5_cache.r5.tumour.npz')
    print('[CELL 2] RUNG-5 panel cache on THIS Drive:', 'FOUND -> instant load' if has else
          'NOT found -> arm(a) will RE-FETCH the atlas (resumable). Use the same account you ran RUNG-5 on to avoid this.')
except Exception as e:
    os.environ['LOGICGATE_CACHE'] = '/content/cancer-recon-apoptosis/data/logicgate/rung5_cache.npz'
    print('[CELL 2] Drive NOT mounted (', type(e).__name__, ') — cache in /content (LOST on disconnect!)')
os.environ['R5_GPU'] = '1'      # use the GPU scoring path (CuPy); clean CPU fallback if no GPU
from scripts.runlog import new_log
RUNLOG = new_log('rung10_andnot3', repo_dir='.')

In [ ]:
#@title Cell 3 — VALIDATE the family build + scorer reuse + checkpoint (CPU, synthetic, instant)
import sys
from scripts.runlog import run_logged
rc = run_logged([sys.executable, '-u', 'scripts/31_andnot3_surface_sweep.py', 'selftest'], RUNLOG)
print('[CELL 3]', '✓ validated' if rc == 0 else '✗ FAILED — stop here')

In [ ]:
#@title Cell 4 — install CELLxGENE Census (CuPy ships with the GPU runtime)
import sys, importlib.util
from scripts.runlog import run_logged
for pkg, pip_name in [('cellxgene_census', 'cellxgene-census'), ('scanpy', 'scanpy')]:
    if importlib.util.find_spec(pkg) is None:
        run_logged([sys.executable, '-m', 'pip', 'install', '-q', pip_name], RUNLOG, label=f'pip install {pip_name}')
print('[CELL 4] ✓ (if Colab asks to RESTART, do it, then Run all — the panel cache makes resume instant)')

In [ ]:
#@title Cell 4b — GPU check + benchmark the bincount the scorer offloads (this rung NEEDS it)
import os, time
try:
    import cupy as cp, numpy as np
    ndev = cp.cuda.runtime.getDeviceCount()
    print('[GPU] CuPy devices:', ndev, '(', cp.cuda.runtime.getDeviceProperties(0)['name'].decode() if ndev else 'none', ')')
    if ndev:
        N = 5_000_000; g = np.random.randint(0, 50000, N); w = (np.random.random(N) > 0.5).astype(float)
        t = time.time(); np.bincount(g, weights=w, minlength=50000); tc = time.time() - t
        gg, gw = cp.asarray(g), cp.asarray(w); cp.cuda.Stream.null.synchronize()
        t = time.time(); cp.bincount(gg, weights=gw, minlength=50000); cp.cuda.Stream.null.synchronize(); tg = time.time() - t
        print(f'[GPU] bincount(5M) CPU {tc*1000:.0f}ms vs GPU {tg*1000:.0f}ms -> {tc/max(tg,1e-6):.0f}x. GPU path ON.')
    else:
        print('[GPU] NO GPU — switch Runtime->T4 GPU, or it will fall back to CPU (much slower for this rung).')
except Exception as e:
    print('[GPU] CuPy unavailable (', type(e).__name__, ') — switch to a T4 GPU runtime for this rung.')

In [ ]:
#@title Cell 5 — REAL run: GPU 3-input surface AND-NOT sweep (RESUMABLE + heartbeat)
#@markdown Loads the RUNG-5 panel (instant if cached on this account, else re-fetches resumably), builds the
#@markdown pruned 3-input AND-NOT family, and scores it on the GPU. A background **[heartbeat]** + per-batch
#@markdown progress show it's alive. **If it disconnects, just RE-RUN THIS CELL** — the per-batch checkpoint
#@markdown on Drive resumes the sweep where it stopped (and the panel cache means no re-fetch).
TOP_POS = 120   #@param {type:"integer"}
N_NEG   = 40    #@param {type:"integer"}
import os, sys
os.environ.update(R10_TOP_POS=str(TOP_POS), R10_N_NEG=str(N_NEG))
from scripts.runlog import run_logged
print(f'[CELL 5] sweep: {TOP_POS} positives x C(.,2) x {N_NEG} NOT-markers; GPU={os.environ.get("R5_GPU")}.')
rc = run_logged([sys.executable, '-u', 'scripts/31_andnot3_surface_sweep.py', 'run'], RUNLOG)
print('[CELL 5]', '✓ done' if rc == 0 else f'✗ exit {rc} — re-run this cell to resume from the last batch')
from IPython.display import Image, display
import json
if os.path.exists('runs/rung10_andnot3/rung10_andnot3.png'):
    display(Image('runs/rung10_andnot3/rung10_andnot3.png'))
if os.path.exists('runs/rung10_andnot3/rung10_andnot3.json'):
    d = json.load(open('runs/rung10_andnot3/rung10_andnot3.json'))
    print('n_safe:', d['n_safe'], 'of', d['family_size'], '| closes gap:', d['closes_gap'])
    print('verdict distribution:', d['verdict_distribution'])

In [ ]:
#@title Cell 6 — bundle the WHOLE run into ONE timestamped zip + download it
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem = os.path.basename(str(RUNLOG)).replace('rung10_andnot3_', '').replace('.log', '')
bundle = f'/content/rung10_run_{stem}.zip'
paths = sorted(set(glob.glob('runs/rung10_andnot3/*.png') + glob.glob('runs/rung10_andnot3/*.json') + [str(RUNLOG)]))
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        if os.path.exists(p):
            z.write(p, arcname=os.path.basename(p)); print('  bundled', p)
print('[CELL 6] ->', bundle)
try:
    from google.colab import files; files.download(bundle)
except Exception as e:
    print('(download skipped:', type(e).__name__, e, ')')

## What RUNG-10 arm (a) establishes
Closes the RUNG-6 plan. If **0 of N** 3-input surface AND-NOT gates are worst-donor-safe (expected), it
confirms — at the bigger search scale where the GPU earns its place — that adding a 3rd *surface* slot doesn't
rescue safety: the bottleneck is the **type** of signal (broadly-expressed surface), not gate arity. The
NOT-slot **must** be a genetic-loss signal (HLA-LOH), exactly the direction RUNG-6/7/8 pursued. If any gate
surprises us by passing, it's a **HYPOTHESIS** — route it through `scripts/23` (FDR + donor-permutation +
bootstrap) and wet-lab agonism before believing it; it is not a cure.

**Honest ceiling:** transcript-level (mRNA ≠ surface protein); pruned search (top positives × broadly-normal
NOT markers), not the full ~1.6×10⁸ exhaustive — stated, not silent; same worst-donor semantics as RUNG-5.